# Previsão do consumo horário de energia (PJM East) com LSTM — janela de 24h × 168h

**Disciplina:** Inteligência Computacional / Machine Learning — CC6NA (2026.2)<br>
**Professora:** Polyana Santos Fonseca Nascimento<br>
**Dupla:** Davi Corrêa e Gabriel Alencar

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaalbu/lstm-energia-pjme/blob/main/notebooks/LSTM_PJME_energia.ipynb)

**Objetivo:** prever o consumo de energia (MW) da **próxima hora** a partir do histórico recente da própria série, comparando
duas janelas deslizantes de entrada — **24 horas** (um dia) e **168 horas** (uma semana) — e verificando se o histórico mais
longo melhora a previsão.

| Seção | Conteúdo | Responsável |
|---|---|---|
| 0 | Declaração de uso de IA Generativa | Gabriel (rascunho) · Davi (revisão final) |
| 1–2 | Configuração do ambiente, dataset e gráfico inicial | Gabriel |
| 3 | Ordenação cronológica e tratamento de lacunas | Gabriel |
| 4 | Divisão treino/validação/teste e normalização | Gabriel |
| 5 | Construção das janelas deslizantes (24h e 168h) | Gabriel |
| 6 | Arquitetura e treino das duas LSTMs | Gabriel |
| 7 | MAE na validação e escolha da janela | Gabriel |
| 8 | Teste: LSTM escolhida × persistência × referência sazonal | Davi |
| 9 | Visualizações complementares do teste | Davi |
| 10 | Conclusão | Davi (com Gabriel) |

## 0. Declaração de uso de IA Generativa

> ✏️ **Revisar antes da entrega** — ajustar a extensão e as medidas de validação ao que a dupla de fato fez.

- **Houve uso de IA Generativa?** Sim.
- **Ferramenta utilizada:** Claude (Anthropic).
- **Finalidade do uso:** apoio na estruturação do notebook segundo a lauda, na escrita do código de pré-processamento,
  construção das janelas e treino das LSTMs, e na redação dos textos explicativos.
- **Extensão aproximada da contribuição:** _preencher (ex.: parte significativa do código das seções 1–7; seções 8–10 feitas pela dupla)_.
- **Medidas de validação:** execução completa do notebook do início ao fim; conferência manual das lacunas encontradas na
  série (horário de verão); verificação de que o normalizador foi ajustado só com o treino e de que as duas janelas usam
  exatamente as mesmas datas-alvo (asserts no código); conferência dos MAE na escala original.

## 1. Configuração do ambiente

O notebook usa apenas bibliotecas que já vêm instaladas no Google Colab (pandas, NumPy, Matplotlib, scikit-learn e
TensorFlow/Keras). Recomenda-se usar um ambiente com **GPU** (*Ambiente de execução → Alterar o tipo de ambiente de
execução → T4 GPU*) — em CPU o treino da janela de 168h leva alguns minutos a mais, mas também funciona.

As sementes aleatórias são fixadas para que os resultados sejam reprodutíveis (podem variar levemente entre CPU e GPU).

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras

SEED = 42
keras.utils.set_random_seed(SEED)

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3})
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "não")

## 2. Dataset

- **Nome:** *Hourly Energy Consumption* — arquivo `PJME_hourly.csv`
- **Fonte:** Kaggle — <https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption>. Os dados são publicados
  originalmente pela **PJM Interconnection**, operadora da rede elétrica de parte do leste dos EUA; `PJME` é a região
  *PJM East*.
- **Variável-alvo:** `PJME_MW` — consumo (carga) de energia elétrica em megawatts.
- **Frequência:** horária, de 01/01/2002 a 03/08/2018 (≈ 145 mil observações).
- **Problema:** prever o consumo da **próxima hora** (um passo à frente) usando apenas os valores anteriores da série.

**Por que esse dataset?** Previsão de carga de curto prazo é um problema real das operadoras (despacho de usinas,
compra de energia). A série é longa, vem de uma fonte confiável e tem ciclos fortes e bem conhecidos — diário (dia × noite),
semanal (dias úteis × fim de semana) e anual (verão e inverno) — o que a torna adequada para testar se uma janela de
uma semana captura algo que uma janela de um dia não captura.

Para que o notebook rode sem precisar de conta no Kaggle, uma cópia do CSV original está na pasta `data/` do repositório.

In [ ]:
CAMINHO_LOCAL = "../data/PJME_hourly.csv"
URL_REPO = "https://raw.githubusercontent.com/Gaalbu/lstm-energia-pjme/main/data/PJME_hourly.csv"

fonte = CAMINHO_LOCAL if os.path.exists(CAMINHO_LOCAL) else URL_REPO
bruto = pd.read_csv(fonte, parse_dates=["Datetime"])
print("Fonte:", fonte)
print(bruto.shape)
bruto.head()

In [ ]:
bruto.describe()

## 3. Ordem cronológica e tratamento de lacunas

O CSV **não vem em ordem cronológica** (começa em 31/12/2002), então o primeiro passo é ordenar pelas datas. Depois
verificamos se a série tem exatamente uma observação por hora.

In [ ]:
bruto = bruto.sort_values("Datetime").reset_index(drop=True)

duplicadas = bruto[bruto["Datetime"].duplicated(keep=False)]
grade_completa = pd.date_range(bruto["Datetime"].min(), bruto["Datetime"].max(), freq="h")
faltantes = grade_completa.difference(bruto["Datetime"])

print(f"Valores nulos em PJME_MW: {bruto['PJME_MW'].isna().sum()}")
print(f"Horários repetidos: {duplicadas['Datetime'].nunique()}")
print(f"Horas ausentes: {len(faltantes)}")
print("\nRepetidos:")
print(duplicadas.to_string(index=False))
print("\nAusentes (amostra):", [str(t) for t in faltantes[:6]], "...")
print("Meses das horas ausentes:", sorted(set(faltantes.month)))

As inconsistências vêm, em sua maioria, do **horário de verão** dos EUA:

- **Horas repetidas** (4 casos, em novembro): quando o relógio volta uma hora, o horário das 02:00 aparece duas vezes.
  Tratamento: substituímos as duas leituras pela **média** delas — fica uma observação por hora.
- **Horas ausentes** (30 casos): a maioria cai nos dias de troca do horário de verão (março/abril e outubro/novembro) e há
  poucas lacunas isoladas em outras datas. Tratamento: **interpolação linear no tempo** entre a hora anterior e a seguinte.
  Como todas as lacunas são de 1 hora e o consumo varia de forma suave de uma hora para a outra, a interpolação é uma estimativa razoável
  e afeta apenas ~0,02% da série.

Depois disso a série fica regular (frequência horária, sem buracos), requisito para que cada janela deslizante de *k*
observações corresponda exatamente a *k* horas.

In [ ]:
serie = (bruto.groupby("Datetime")["PJME_MW"].mean()   # junta horários repetidos pela média
              .reindex(grade_completa)                   # cria as horas ausentes como NaN
              .interpolate(method="time"))               # preenche por interpolação linear no tempo
serie.index.name = "Datetime"
serie.name = "PJME_MW"

maior_lacuna = (pd.Series(faltantes).diff() != pd.Timedelta("1h")).cumsum().value_counts().max()
assert serie.isna().sum() == 0 and serie.index.is_monotonic_increasing and serie.index.freq == "h"
print(f"Série final: {len(serie)} horas | maior lacuna consecutiva preenchida: {maior_lacuna} h")

### Gráfico inicial da série

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(13, 10))

serie.plot(ax=ax[0], lw=0.3, color="tab:blue")
serie.resample("W").mean().plot(ax=ax[0], color="black", lw=1.2, label="média semanal")
ax[0].set_title("PJME — consumo horário completo (2002–2018)")
ax[0].set_ylabel("MW"); ax[0].legend()

semanas = serie["2017-07-03":"2017-07-16"]
semanas.plot(ax=ax[1], color="tab:orange")
ax[1].set_title("Zoom: duas semanas de julho/2017 — ciclo diário e diferença dias úteis × fim de semana")
ax[1].set_ylabel("MW")

perfil = serie.groupby([serie.index.dayofweek, serie.index.hour]).mean().unstack(0)
perfil.columns = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
perfil.plot(ax=ax[2])
ax[2].set_title("Perfil médio por hora do dia, para cada dia da semana")
ax[2].set_xlabel("hora do dia"); ax[2].set_ylabel("MW médio")

plt.tight_layout(); plt.show()

O gráfico mostra (1) uma **sazonalidade anual** com picos no verão (ar-condicionado) e um pico menor no inverno;
(2) um **ciclo diário** muito marcado — mínimo de madrugada e máximo à tarde/início da noite; e (3) um **padrão semanal**:
sábado e domingo têm consumo visivelmente menor que os dias úteis. Esses ciclos orientam a escolha das janelas na seção 5.

## 4. Divisão cronológica e normalização

A divisão é feita **pelas datas-alvo** (o instante que se quer prever), em ordem cronológica e sem embaralhar:

| Partição | Proporção da série | Uso |
|---|---|---|
| Treino | primeiros ~65% | ajustar os pesos da LSTM e o normalizador |
| Validação | ~15% seguintes | *early stopping* e **escolha da janela** (24h × 168h) |
| Teste | **últimos 20%** | avaliação final contra as referências |

Para que as duas configurações sejam avaliadas **nas mesmas datas-alvo**, o primeiro alvo possível é a hora 168 da série
(a janela maior precisa de 168 horas anteriores). Assim, a janela de 24h não ganha alvos "extras" no começo do treino.
A janela de entrada de um alvo pode conter horas da partição anterior (por exemplo, o primeiro alvo da validação usa as
últimas horas do treino como entrada) — isso não é vazamento, porque são valores passados, conhecidos no momento da
previsão.

A normalização Min-Max (escala 0–1) é **ajustada apenas com os valores do período de treino** e depois aplicada à série
inteira.

In [ ]:
JANELAS = [24, 168]
JANELA_MAX = max(JANELAS)

n = len(serie)
i_val = int(n * 0.65)    # primeiro alvo da validação
i_teste = int(n * 0.80)  # primeiro alvo do teste  -> últimos 20% da série

idx_alvos = {
    "treino": np.arange(JANELA_MAX, i_val),
    "validacao": np.arange(i_val, i_teste),
    "teste": np.arange(i_teste, n),
}
datas = {p: serie.index[i] for p, i in idx_alvos.items()}

for p, d in datas.items():
    print(f"{p:<10} {d[0]}  ->  {d[-1]}   ({len(d):>6} alvos, {len(d)/n:.1%} da série)")

In [ ]:
scaler = MinMaxScaler()
scaler.fit(serie.iloc[:i_val].to_frame())          # somente o período de treino
serie_norm = scaler.transform(serie.to_frame()).ravel().astype("float32")

print(f"Min/max usados pelo scaler (treino): {scaler.data_min_[0]:.0f} / {scaler.data_max_[0]:.0f} MW")
print(f"Faixa normalizada em validação+teste: {serie_norm[i_val:].min():.3f} a {serie_norm[i_val:].max():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))
cores = {"treino": "tab:blue", "validacao": "tab:orange", "teste": "tab:green"}
for p, idx in idx_alvos.items():
    ax.plot(serie.index[idx], serie.values[idx], lw=0.3, color=cores[p], label=p)
ax.set_title("Divisão cronológica por data-alvo"); ax.set_ylabel("MW")
ax.legend(loc="upper left", markerscale=10); plt.show()

## 5. Janelas deslizantes de entrada

Usamos a **janela deslizante** de tamanho fixo: para prever o valor da hora *t*, a entrada são as *k* horas imediatamente
anteriores, $x_{t-k}, \dots, x_{t-1}$. A cada novo exemplo a janela avança **uma observação** (o início e o fim andam juntos).

**Tamanhos escolhidos (dados horários):**

- **k = 24 horas (um dia):** cobre um ciclo diário completo — a rede vê o mesmo horário do dia anterior e todo o formato
  da curva de consumo do último dia.
- **k = 168 horas (uma semana, 7 × 24):** cobre um ciclo semanal completo — além do dia anterior, a rede vê o mesmo
  horário e o mesmo dia da semana anterior, o que pode ajudar nas transições entre dias úteis e fim de semana
  (ex.: prever uma segunda-feira olhando só para um domingo tende a subestimar o consumo).

A pergunta do trabalho é se esse histórico extra (7× maior) compensa o custo de uma sequência mais longa.

In [ ]:
def criar_janelas(serie_norm, idx_alvos, k):
    # X[i] = os k valores anteriores ao alvo idx_alvos[i];  y[i] = valor no alvo
    offsets = np.arange(-k, 0)
    X = serie_norm[idx_alvos[:, None] + offsets[None, :]]
    y = serie_norm[idx_alvos]
    return X[..., None], y          # (amostras, k, 1 variável), (amostras,)

dados = {k: {p: criar_janelas(serie_norm, idx, k) for p, idx in idx_alvos.items()} for k in JANELAS}

for k in JANELAS:
    print(f"janela {k:>3}h:", {p: X.shape for p, (X, _) in dados[k].items()})

# conferências: mesmos alvos nas duas janelas e janela termina na hora anterior ao alvo
for p in idx_alvos:
    assert np.array_equal(dados[24][p][1], dados[168][p][1])
    assert np.array_equal(dados[24][p][0][:, :, 0], dados[168][p][0][:, -24:, 0])
i0 = idx_alvos["validacao"][0]
assert dados[24]["validacao"][0][0, -1, 0] == serie_norm[i0 - 1]
print("OK: as duas configurações têm exatamente as mesmas datas-alvo em todas as partições.")

## 6. Arquitetura e treino das LSTMs

As duas configurações usam **a mesma arquitetura e os mesmos hiperparâmetros** — a única diferença é o comprimento da
sequência de entrada:

- Entrada: sequência de *k* passos com 1 variável (consumo normalizado).
- Camada **LSTM com 64 unidades** (retorna só o último estado oculto).
- Camada **Dense(1)** linear → consumo previsto para a próxima hora (normalizado).
- Perda MSE, otimizador Adam (lr = 0,001), lotes de 256 exemplos, até 60 épocas.
- **Early stopping** monitorando a perda na validação (paciência 5), restaurando os melhores pesos.

O MSE é usado no treino por penalizar mais os erros grandes (picos de carga); a comparação entre janelas é feita com o
**MAE na escala original (MW)**, como pede a lauda.

In [ ]:
UNIDADES = 64
EPOCAS = 60
LOTE = 256
TAXA_APRENDIZADO = 1e-3
PACIENCIA = 5

def criar_lstm(k):
    modelo = keras.Sequential([
        keras.Input(shape=(k, 1)),
        keras.layers.LSTM(UNIDADES),
        keras.layers.Dense(1),
    ], name=f"lstm_janela_{k}h")
    modelo.compile(optimizer=keras.optimizers.Adam(TAXA_APRENDIZADO), loss="mse", metrics=["mae"])
    return modelo

criar_lstm(24).summary()

In [ ]:
modelos, historicos = {}, {}

for k in JANELAS:
    keras.utils.set_random_seed(SEED)       # mesma inicialização para as duas configurações
    X_tr, y_tr = dados[k]["treino"]
    X_va, y_va = dados[k]["validacao"]
    modelo = criar_lstm(k)
    parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True)
    print(f"\n=== Treinando janela de {k}h ===")
    hist = modelo.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=EPOCAS, batch_size=LOTE,
                      callbacks=[parada], verbose=2)
    modelos[k], historicos[k] = modelo, hist.history
    print(f"melhor época: {int(np.argmin(hist.history['val_loss'])) + 1} de {len(hist.history['val_loss'])}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for a, k in zip(ax, JANELAS):
    a.plot(historicos[k]["loss"], label="treino")
    a.plot(historicos[k]["val_loss"], label="validação")
    a.set_yscale("log"); a.set_title(f"Curva de perda (MSE normalizado) — janela {k}h")
    a.set_xlabel("época"); a.legend()
plt.tight_layout(); plt.show()

## 7. Comparação das janelas na validação (MAE em MW)

As previsões são convertidas de volta para MW com o mesmo `scaler` (transformação inversa) e o MAE é calculado na
**escala original**.

In [ ]:
def para_mw(valores_norm):
    return scaler.inverse_transform(np.asarray(valores_norm).reshape(-1, 1)).ravel()

def prever_mw(modelo, X):
    return para_mw(modelo.predict(X, batch_size=2048, verbose=0))

def mae(real, previsto):
    return float(np.mean(np.abs(np.asarray(real) - np.asarray(previsto))))

y_val_mw = serie.values[idx_alvos["validacao"]]
prev_val = {k: prever_mw(modelos[k], dados[k]["validacao"][0]) for k in JANELAS}
mae_val = {k: mae(y_val_mw, prev_val[k]) for k in JANELAS}

janela_escolhida = min(mae_val, key=mae_val.get)

tabela_val = pd.DataFrame({
    "Janela": [f"{k} h" for k in JANELAS],
    "MAE validação (MW)": [round(mae_val[k], 1) for k in JANELAS],
    "MAE / consumo médio": [f"{mae_val[k] / y_val_mw.mean():.2%}" for k in JANELAS],
    "Épocas treinadas": [len(historicos[k]["loss"]) for k in JANELAS],
    "Escolhida": ["✔" if k == janela_escolhida else "" for k in JANELAS],
})
display(tabela_val) if "display" in dir() else print(tabela_val)
print(f"\nJanela escolhida pela validação: {janela_escolhida} h")

In [ ]:
inicio, fim = "2014-07-07", "2014-07-20"     # duas semanas de verão (período de validação)
mask = (datas["validacao"] >= inicio) & (datas["validacao"] < fim)

fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax[0].plot(datas["validacao"][mask], y_val_mw[mask], color="black", lw=1.4, label="real")
for k, cor in zip(JANELAS, ["tab:blue", "tab:red"]):
    ax[0].plot(datas["validacao"][mask], prev_val[k][mask], color=cor, lw=1, alpha=0.85,
               label=f"LSTM {k}h (MAE período = {mae(y_val_mw[mask], prev_val[k][mask]):.0f} MW)")
    ax[1].plot(datas["validacao"][mask], prev_val[k][mask] - y_val_mw[mask], color=cor, lw=0.9, label=f"erro {k}h")
ax[0].set_title("Validação — real × previsões das duas janelas (zoom em duas semanas)"); ax[0].set_ylabel("MW"); ax[0].legend()
ax[1].axhline(0, color="black", lw=0.8); ax[1].set_ylabel("previsto − real (MW)"); ax[1].legend()
ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
plt.tight_layout(); plt.show()

In [ ]:
erro_abs = pd.DataFrame({f"{k}h": np.abs(prev_val[k] - y_val_mw) for k in JANELAS}, index=datas["validacao"])
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
erro_abs.groupby(erro_abs.index.hour).mean().plot(ax=ax[0], marker="o", ms=3)
ax[0].set_title("MAE na validação por hora do dia"); ax[0].set_xlabel("hora"); ax[0].set_ylabel("MW")
por_dia = erro_abs.groupby(erro_abs.index.dayofweek).mean()
por_dia.index = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
por_dia.plot.bar(ax=ax[1], rot=0)
ax[1].set_title("MAE na validação por dia da semana"); ax[1].set_ylabel("MW")
plt.tight_layout(); plt.show()

### Leitura dos resultados da validação

> ✏️ **Escrever depois de rodar no Colab:** usar a tabela acima e a frase impressa na célula abaixo para comentar qual
> janela teve menor MAE, a diferença percentual, as curvas de perda (se houve sinal de *overfitting* ou se a rede ainda
> estava melhorando) e em quais horas/dias o erro foi maior. Numa execução preliminar (30 épocas, CPU), a janela de **168h**
> teve MAE ≈ 275 MW contra ≈ 308 MW da de 24h na validação.

In [ ]:
melhor, pior = janela_escolhida, [k for k in JANELAS if k != janela_escolhida][0]
ganho = (mae_val[pior] - mae_val[melhor]) / mae_val[pior]
print(f"A janela de {melhor}h teve MAE de {mae_val[melhor]:.1f} MW contra {mae_val[pior]:.1f} MW da janela de {pior}h "
      f"({ganho:.1%} menor). Ela é a configuração levada para o teste.")

---
### 🔁 Passagem para a parte do Davi

Tudo o que as seções 8–10 precisam já está em memória:

| Variável | O que é |
|---|---|
| `janela_escolhida` | 24 ou 168, escolhida pelo MAE da validação |
| `modelos[janela_escolhida]` | LSTM treinada da janela escolhida |
| `dados[janela_escolhida]["teste"]` | `(X_teste, y_teste_normalizado)` |
| `idx_alvos["teste"]`, `datas["teste"]` | posições e datas-alvo do teste |
| `serie` | série completa em MW (já limpa) — serve para as referências |
| `prever_mw(modelo, X)`, `mae(real, previsto)`, `para_mw(...)` | funções auxiliares |

Referências no teste (para o alvo na posição `t` de `serie`):
- **Persistência:** `serie.values[t - 1]` (último valor observado).
- **Sazonal diária:** `serie.values[t - 24]` (mesma hora do dia anterior) — ciclo diário justificado pelo gráfico da seção 2.
- (opcional) **Sazonal semanal:** `serie.values[t - 168]` (mesma hora da semana anterior).
---

## 8. Teste: LSTM escolhida × referências — *parte do Davi*

A LSTM da janela escolhida na seção 7 é avaliada **uma única vez** no conjunto de teste (os últimos 20% da série, que não
foram usados nem no treino nem na escolha da janela). Ela é comparada com previsões de **referência**, que não aprendem
nada e servem de régua: se a LSTM não for melhor que elas, o modelo não está agregando valor.

| Método | Previsão para a hora *t* | Justificativa |
|---|---|---|
| LSTM (janela escolhida) | saída da rede a partir das *k* horas anteriores | modelo do trabalho |
| **Persistência** | $\hat{x}_t = x_{t-1}$ | último valor observado (referência exigida pela lauda) |
| **Sazonal diária** | $\hat{x}_t = x_{t-24}$ | mesma hora do dia anterior — o ciclo de 24h é o padrão mais forte do gráfico da seção 2 |
| **Sazonal semanal** | $\hat{x}_t = x_{t-168}$ | mesma hora e mesmo dia da semana anterior — ciclo semanal (dias úteis × fim de semana) |

A sazonal diária é a referência sazonal principal (ciclo dia × noite); a semanal foi incluída porque o gráfico de perfil
médio da seção 2 também mostra um ciclo de 7 dias, e ela é a referência "natural" da janela de 168h.

Todos os métodos são avaliados **nas mesmas datas-alvo** (`idx_alvos["teste"]`) e com o **MAE na escala original (MW)**.
As referências são calculadas sobre a mesma série limpa (seção 3) que alimenta a LSTM, e todas usam apenas valores
**anteriores** à hora prevista — nenhum método tem acesso a informação do futuro.

In [ ]:
try:
    from IPython.display import display
except ImportError:      # fora do Jupyter/Colab
    display = print

k = janela_escolhida
idx_t = idx_alvos["teste"]
datas_t = datas["teste"]
y_teste_mw = serie.values[idx_t]

# conferência: o alvo das janelas, convertido de volta para MW, é a própria série nas datas-alvo do teste
assert np.allclose(para_mw(dados[k]["teste"][1]), y_teste_mw, atol=1.0)

NOME_LSTM = f"LSTM {k}h"
prev_teste = {
    NOME_LSTM: prever_mw(modelos[k], dados[k]["teste"][0]),
    "Persistência (t−1)": serie.values[idx_t - 1],
    "Sazonal diária (t−24)": serie.values[idx_t - 24],
    "Sazonal semanal (t−168)": serie.values[idx_t - 168],
}
assert all(len(p) == len(idx_t) for p in prev_teste.values())   # mesmas datas-alvo para todos

mae_teste = {m: mae(y_teste_mw, p) for m, p in prev_teste.items()}

tabela_teste = pd.DataFrame({
    "Método": list(mae_teste),
    "MAE teste (MW)": [round(v, 1) for v in mae_teste.values()],
    "MAE / consumo médio": [f"{v / y_teste_mw.mean():.2%}" for v in mae_teste.values()],
    "Redução do MAE com a LSTM": ["—" if m == NOME_LSTM else f"{1 - mae_teste[NOME_LSTM] / v:.1%}"
                                  for m, v in mae_teste.items()],
}).sort_values("MAE teste (MW)").reset_index(drop=True)

print(f"Teste: {datas_t[0]} -> {datas_t[-1]}  ({len(idx_t)} horas previstas, consumo médio {y_teste_mw.mean():.0f} MW)")
display(tabela_teste)

### Gráfico do teste: valores reais × LSTM × referências

O teste tem mais de 29 mil horas; para que as curvas fiquem legíveis, o gráfico principal mostra **duas semanas de verão
de 2017** (incluindo o feriado de 4 de julho). Os MAE da legenda e do gráfico de barras são calculados sobre **todo o
teste**, em MW. O painel inferior direito aproxima dois dias para mostrar o comportamento hora a hora.

In [ ]:
ESTILOS = {
    NOME_LSTM: dict(color="tab:red", lw=1.4),
    "Persistência (t−1)": dict(color="tab:gray", lw=1.1, ls="--"),
    "Sazonal diária (t−24)": dict(color="tab:green", lw=1.1, ls=":"),
    "Sazonal semanal (t−168)": dict(color="tab:purple", lw=1.1, ls="-."),
}
ZOOM = ("2017-07-03", "2017-07-17")        # duas semanas (segunda a segunda), inclui o feriado de 04/07
ZOOM_DIAS = ("2017-07-11", "2017-07-13")   # dois dias úteis comuns

def plotar_periodo(ax, inicio, fim, legenda_mae=True):
    m_ = (datas_t >= inicio) & (datas_t < fim)
    ax.plot(datas_t[m_], y_teste_mw[m_], color="black", lw=2.2, label="real")
    for met, prev in prev_teste.items():
        rotulo = f"{met} — MAE teste = {mae_teste[met]:.0f} MW" if legenda_mae else met
        ax.plot(datas_t[m_], prev[m_], label=rotulo, **ESTILOS[met])
    ax.set_ylabel("MW")

fig = plt.figure(figsize=(14, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[1.3, 1], width_ratios=[1, 1.6])

ax0 = fig.add_subplot(gs[0, :])
plotar_periodo(ax0, *ZOOM)
ax0.set_title("Teste — valores reais × previsões (03/07 a 16/07/2017)")
ax0.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
ax0.legend(fontsize=9, ncol=2, loc="upper left")

ax1 = fig.add_subplot(gs[1, 0])
ordem = sorted(mae_teste, key=mae_teste.get)
barras = ax1.barh(ordem, [mae_teste[m] for m in ordem], color=[ESTILOS[m]["color"] for m in ordem])
ax1.bar_label(barras, fmt="%.0f MW", padding=3)
ax1.invert_yaxis(); ax1.set_xlabel("MAE no teste (MW)"); ax1.set_title("MAE de cada método — teste completo")
ax1.set_xlim(0, max(mae_teste.values()) * 1.25)

ax2 = fig.add_subplot(gs[1, 1])
plotar_periodo(ax2, *ZOOM_DIAS, legenda_mae=False)
ax2.set_title("Zoom: 11 e 12/07/2017, hora a hora")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m %Hh"))

plt.tight_layout(); plt.show()

### Leitura dos resultados do teste

> ✏️ **Conferir depois de rodar no Colab:** os números citados abaixo vêm de uma execução preliminar (CPU, 15 épocas). Com as 60 épocas e GPU os valores mudam um pouco (tendem a cair); atualizar com as saídas das células acima.

- A **LSTM de 168h foi, com folga, o melhor método no teste**: MAE ≈ 304 MW, cerca de **1% do consumo médio**
  (≈ 31 mil MW). Isso representa uma redução de ≈ 72% no erro em relação à persistência (≈ 1.070 MW) e de ≈ 86% em relação
  à sazonal diária (≈ 2.180 MW).
- A **persistência** é a referência mais difícil de bater num horizonte de 1 hora, porque o consumo muda pouco de uma
  hora para a seguinte. Mas ela erra sistematicamente nas **rampas** (subida da manhã e descida da noite): como repete o
  valor anterior, a curva dela fica sempre **uma hora atrasada** — o que aparece claramente no zoom de 11 e 12/07.
  A LSTM acompanha a mudança de direção da curva porque aprendeu o formato do ciclo diário.
- As **referências sazonais** acertam o *formato* da curva, mas erram o *nível*: o consumo muda de um dia (ou de uma semana)
  para o outro conforme o clima. No feriado de 04/07, por exemplo, a sazonal diária repete o pico do dia útil anterior e
  superestima o consumo em vários milhares de MW.
- A **sazonal semanal** foi a pior (≈ 3.390 MW): em uma semana o tempo muda bastante, e copiar a semana anterior traz
  mais erro do que o ganho de alinhar o mesmo dia da semana.

## 9. Visualizações complementares do teste — *parte do Davi*

O MAE resume o erro em um único número. As visualizações abaixo mostram **onde** e **quando** cada método erra:
(9.1) a qualidade da LSTM ao longo de toda a faixa de consumo e a distribuição dos erros; (9.2) o erro por hora do dia
e por dia da semana; (9.3) a estabilidade do erro mês a mês ao longo dos ~3 anos de teste; e (9.4) o comportamento em
feriados e nos dias de maior erro.

### 9.1 Dispersão real × previsto e distribuição dos erros

In [ ]:
erros = pd.DataFrame({m: p - y_teste_mw for m, p in prev_teste.items()}, index=datas_t)   # previsto − real

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

hb = ax[0].hexbin(y_teste_mw, prev_teste[NOME_LSTM], gridsize=70, bins="log", cmap="viridis", mincnt=1)
lim = [min(y_teste_mw.min(), prev_teste[NOME_LSTM].min()), max(y_teste_mw.max(), prev_teste[NOME_LSTM].max())]
ax[0].plot(lim, lim, "r--", lw=1, label="previsão perfeita (y = x)")
fig.colorbar(hb, ax=ax[0], label="nº de horas (escala log)")
ax[0].set_xlabel("real (MW)"); ax[0].set_ylabel("previsto (MW)"); ax[0].legend(loc="upper left")
ax[0].set_title(f"{NOME_LSTM} no teste — real × previsto")

LIM_ERRO = 4000
faixas = np.linspace(-LIM_ERRO, LIM_ERRO, 161)
for m in [NOME_LSTM, "Persistência (t−1)", "Sazonal diária (t−24)"]:
    ax[1].hist(erros[m].clip(-LIM_ERRO, LIM_ERRO), bins=faixas, histtype="step", lw=1.5,
               color=ESTILOS[m]["color"], label=f"{m}  (viés médio {erros[m].mean():+.0f} MW)")
ax[1].axvline(0, color="black", lw=0.8)
ax[1].set_xlabel(f"erro = previsto − real (MW; valores além de ±{LIM_ERRO} agrupados nas bordas)")
ax[1].set_ylabel("nº de horas"); ax[1].legend(fontsize=9); ax[1].set_title("Distribuição dos erros no teste")

plt.tight_layout(); plt.show()

q = erros.abs().quantile([0.5, 0.9, 0.99]).round(0)
q.index = ["mediana do erro absoluto", "percentil 90", "percentil 99"]
display(q)

- Na dispersão, os pontos ficam concentrados sobre a reta *y = x* em **toda a faixa de consumo** (≈ 19 mil a 57 mil MW),
  inclusive nos picos de verão: não há saturação nos extremos. O maior valor do teste (≈ 56,6 mil MW) fica dentro da
  faixa vista no treino (máximo de ≈ 62 mil MW usado pelo `MinMaxScaler`).
- Os erros da LSTM são bem mais concentrados em torno de zero: **mediana ≈ 213 MW**, percentil 90 ≈ 700 MW e percentil
  99 ≈ 1.300 MW. Na persistência, a mediana é ≈ 850 MW e o percentil 99 chega a ≈ 3.460 MW.
- O viés médio da LSTM é pequeno (≈ +25 MW, leve tendência a superestimar), desprezível frente ao MAE.

### 9.2 Erro por hora do dia e por dia da semana

In [ ]:
erro_abs_t = erros.abs()
DIAS_SEMANA = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

por_hora = erro_abs_t.groupby(erro_abs_t.index.hour).mean()
for m in por_hora:
    ax[0].plot(por_hora.index, por_hora[m], marker="o", ms=3, label=m, **ESTILOS[m])
ax[0].set_yscale("log"); ax[0].set_xticks(range(0, 24, 2))
ax[0].set_title("MAE no teste por hora do dia"); ax[0].set_xlabel("hora"); ax[0].set_ylabel("MW (escala log)")
ax[0].legend(fontsize=8)

por_dia = erro_abs_t.groupby(erro_abs_t.index.dayofweek).mean()
por_dia.index = DIAS_SEMANA
por_dia.plot.bar(ax=ax[1], rot=0, color=[ESTILOS[m]["color"] for m in por_dia.columns], legend=False, logy=True)
ax[1].set_title("MAE no teste por dia da semana"); ax[1].set_ylabel("MW (escala log)")

plt.tight_layout(); plt.show()
display(por_dia.round(0).T)

- **Por hora do dia:** a LSTM erra menos de madrugada (≈ 100–150 MW entre 1h e 4h, quando o consumo é estável) e mais nas
  **rampas** — na subida da manhã (≈ 630 MW às 7h) e no início da noite (≈ 540 MW às 19h), horários em que a intensidade da
  variação depende do clima e da rotina daquele dia. A persistência tem o mesmo padrão, só que muito amplificado
  (≈ 2.200 MW às 7h e às 23h).
- **Por dia da semana:** o erro da LSTM é praticamente uniforme (≈ 290–320 MW), um pouco maior no sábado, domingo e
  segunda. Já a sazonal diária piora muito na **segunda-feira** (≈ 3.350 MW, porque copia o domingo) e no **sábado**
  (≈ 2.840 MW, porque copia a sexta). Isso confirma o ciclo semanal da série — justamente a informação que a janela de
  168h coloca à disposição da LSTM.

### 9.3 Erro mês a mês ao longo do teste

In [ ]:
mes_completo = erro_abs_t.resample("MS").size() >= 24 * 28     # descarta o 1º e o último mês, incompletos
mensal = erro_abs_t.resample("MS").mean()[mes_completo]
consumo_mensal = pd.Series(y_teste_mw, index=datas_t).resample("MS").mean()[mes_completo]

fig, ax = plt.subplots(figsize=(14, 4.5))
for m in mensal:
    ax.plot(mensal.index, mensal[m], marker="o", ms=3, label=m, **ESTILOS[m])
ax.set_yscale("log"); ax.set_ylabel("MAE mensal (MW, escala log)")
ax2 = ax.twinx()
ax2.fill_between(consumo_mensal.index, consumo_mensal.values, color="tab:blue", alpha=0.08, step="mid")
ax2.set_ylabel("consumo médio mensal (MW)", color="tab:blue"); ax2.grid(False)
ax2.set_ylim(consumo_mensal.min() * 0.9, consumo_mensal.max() * 1.05)
ax.set_title("MAE mensal no teste (linhas) e consumo médio do mês (área azul) — meses completos")
ax.legend(fontsize=8, ncol=4, loc="upper left")
plt.tight_layout(); plt.show()

verao = mensal.index.month.isin([6, 7, 8])
print(f"MAE médio da {NOME_LSTM}: meses de verão (jun–ago) = {mensal.loc[verao, NOME_LSTM].mean():.0f} MW | "
      f"demais meses = {mensal.loc[~verao, NOME_LSTM].mean():.0f} MW")

- O MAE mensal da LSTM fica **estável ao longo dos ~3 anos de teste** (≈ 240–400 MW), sem tendência de piora com o tempo:
  o modelo treinado com dados até 2012 continua válido para 2015–2018.
- Não há diferença relevante entre o verão e os demais meses (≈ 302 × ≈ 305 MW, linha impressa acima). As referências,
  ao contrário, pioram bastante nos meses de alto consumo (verão e inverno rigoroso), quando o consumo varia mais de um
  dia para o outro.
- O pior mês da LSTM foi **janeiro de 2018** (≈ 400 MW), mês de consumo atipicamente alto para o inverno (ver 9.4).

### 9.4 Feriados e dias de maior erro

A LSTM recebe apenas a própria série, sem nenhuma informação de calendário. Por isso vale verificar o desempenho nos
**feriados federais dos EUA** (calendário `USFederalHolidayCalendar` do pandas), quando o consumo cai como em um fim de
semana, e listar os dias de maior erro.

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar

feriados = USFederalHolidayCalendar().holidays(datas_t.min().normalize(), datas_t.max())
dia_t = datas_t.normalize()
grupo = np.select([dia_t.isin(feriados), dia_t.isin(feriados + pd.Timedelta(days=1))],
                  ["feriado", "dia seguinte ao feriado"], "demais dias")

tabela_feriados = erro_abs_t.groupby(grupo).mean().reindex(["demais dias", "feriado", "dia seguinte ao feriado"]).round(0)
tabela_feriados.insert(0, "horas", pd.Series(grupo).value_counts())
print(f"{len(feriados)} feriados federais no período de teste")
display(tabela_feriados)

# dias completos (24 h) com maior MAE da LSTM
horas_no_dia = erro_abs_t.resample("D").size()
mae_dia = erro_abs_t.resample("D").mean()[horas_no_dia == 24]
top = mae_dia[NOME_LSTM].nlargest(10)
tabela_top = pd.DataFrame({
    "Data": top.index.strftime("%d/%m/%Y"),
    "Dia": [DIAS_SEMANA[d.dayofweek] for d in top.index],
    "Feriado federal": ["sim" if d in feriados else "" for d in top.index],
    "Consumo médio (MW)": pd.Series(y_teste_mw, index=datas_t).resample("D").mean().loc[top.index].round(0).values,
    f"MAE {NOME_LSTM}": top.round(0).values,
    "MAE persistência": mae_dia.loc[top.index, "Persistência (t−1)"].round(0).values,
    "MAE sazonal diária": mae_dia.loc[top.index, "Sazonal diária (t−24)"].round(0).values,
})
print("\nDez dias do teste com maior MAE da LSTM:")
display(tabela_top)

- **Feriados:** a LSTM praticamente não é afetada (≈ 308 MW nos feriados e ≈ 330 MW no dia seguinte, contra ≈ 303 MW nos
  demais dias). Como ela prevê só uma hora à frente, as primeiras horas do próprio feriado já mostram o consumo mais baixo e
  a rede se ajusta. A sazonal diária, por outro lado, piora bastante (≈ 2.640 MW no feriado e ≈ 3.030 MW no dia seguinte,
  contra ≈ 2.150 MW), porque copia um dia de comportamento diferente.
- **Dias de maior erro:** cinco dos dez piores dias da LSTM estão entre **01 e 05/01/2018**, com consumo médio acima de
  40 mil MW, nível incomum para o inverno — período de frio intenso no leste dos EUA. Sem dados de temperatura, a rede
  só percebe a mudança pela própria série. Também aparece **12/03/2017**, domingo de início do horário de verão (hora
  ausente preenchida por interpolação na seção 3 e curva diária deslocada em uma hora).
- Mesmo nesses dias difíceis, o erro da LSTM (≈ 530–660 MW) ficou abaixo do da persistência e do da sazonal diária.

In [ ]:
# resumo numérico usado na conclusão (gerado a partir das saídas acima)
outra = [j for j in JANELAS if j != janela_escolhida][0]
print(f"Validação: janela {janela_escolhida}h = {mae_val[janela_escolhida]:.1f} MW  |  janela {outra}h = {mae_val[outra]:.1f} MW"
      f"  -> {janela_escolhida}h com MAE {1 - mae_val[janela_escolhida] / mae_val[outra]:.1%} menor")
print(f"Teste ({NOME_LSTM}):")
for m in sorted(mae_teste, key=mae_teste.get):
    extra = "" if m == NOME_LSTM else f"  (LSTM reduz o MAE em {1 - mae_teste[NOME_LSTM] / mae_teste[m]:.1%})"
    print(f"  {m:<24} {mae_teste[m]:8.1f} MW{extra}")

## 10. Conclusão — *parte do Davi (com Gabriel)*

> ✏️ **Conferir depois de rodar no Colab:** os números citados abaixo vêm de uma execução preliminar (CPU, 15 épocas). Com as 60 épocas e GPU os valores mudam um pouco (tendem a cair); atualizar com as saídas das células acima.

**1. A janela mais longa ajudou na validação?** Sim. A janela de **168h** teve MAE ≈ 321 MW na validação, contra
≈ 385 MW da janela de 24h — um erro ≈ 16% menor — e por isso foi a escolhida. Como as duas configurações tinham a mesma
arquitetura, os mesmos hiperparâmetros e as mesmas datas-alvo, a diferença vem apenas do histórico: com uma semana de
entrada, a rede enxerga o mesmo horário e o mesmo dia da semana anterior, o que ajuda nas transições entre dias úteis e
fim de semana (o efeito que a seção 9.2 mostra na referência sazonal diária). O custo é um treino bem mais lento: a
sequência é 7× maior e, na CPU, cada época levou cerca de 6× mais tempo.

**2. Como a LSTM se saiu no teste em relação às referências?** Muito melhor que todas: MAE ≈ **304 MW** (≈ 1% do consumo
médio), contra ≈ 1.070 MW da persistência, ≈ 2.180 MW da sazonal diária e ≈ 3.390 MW da sazonal semanal — reduções de
≈ 72%, 86% e 91%. O MAE do teste ficou próximo ao da validação, sem sinal de *overfitting*, e o erro se manteve estável
ao longo dos três anos de teste, nos meses de verão e nos feriados (seção 9). Os maiores erros se concentram nas rampas
da manhã e da noite e em dias de clima extremo.

**Limitações**
- O modelo é **univariado**: usa só o próprio consumo, sem temperatura (o principal fator externo, como mostra o início
  de janeiro de 2018) e sem variáveis de calendário (dia da semana, feriados).
- A previsão é de **um único passo** (1 hora), o horizonte mais fácil; na operação real, a carga costuma ser prevista
  para o dia seguinte inteiro.
- Os resultados vêm de **uma única execução** (uma semente); a variação entre treinos não foi medida.
- Só foram comparadas duas janelas, com hiperparâmetros fixos (sem busca).

**Possíveis melhorias:** incluir temperatura e variáveis de calendário como entradas; prever vários passos à frente
(próximas 24h); repetir o treino com várias sementes e reportar média ± desvio do MAE; testar janelas intermediárias
(48h, 72h) e arquiteturas com mais camadas.